# Commodity Data Collector
### Prices (FRED) · News (SerpAPI) · FinBERT Embeddings & Sentiment

Covers **wheat**, **corn**, and **oil** from 2010-01-01 → 2026-03-31.

Output folder structure:
```
data/
  wheat/
    wheat_prices.csv
    wheat_news.csv
    wheat_news_embeddings.pt
    wheat_news_sentiment.csv
  corn/
    corn_prices.csv
    corn_news.csv
    corn_news_embeddings.pt
    corn_news_sentiment.csv
  oil/
    oil_prices.csv
    oil_news.csv
    oil_news_embeddings.pt
    oil_news_sentiment.csv
```

In [13]:
import subprocess, sys
_pkgs = ['fredapi', 'python-dotenv', 'google-search-results',
         'transformers', 'torch', 'scikit-learn', 'pandas',
         'numpy', 'tqdm', 'accelerate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _pkgs,
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('✓ Dependencies installed')

✓ Dependencies installed


---
## 1 · Configuration

In [ ]:
import os, re, json, time, random, warnings, copy
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from fredapi import Fred
from serpapi import GoogleSearch
from tqdm import tqdm
from sklearn.decomposition import PCA
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings('ignore')
load_dotenv(dotenv_path=Path('.env'))

# ── API keys ──────────────────────────────────────────────────────────────────
FRED_API_KEY = os.getenv('FRED')
SERP_API_KEY = os.getenv('SERP_API')
assert FRED_API_KEY, 'FRED key missing from .env'
assert SERP_API_KEY, 'SERP_API key missing from .env'
print(f'✓ API keys loaded (FRED, SerpAPI)')

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = '2010-01-01'
END_DATE   = '2026-03-31'
YEARS      = list(range(2010, 2027))

# ── Commodity configuration ───────────────────────────────────────────────────
# 6 queries per commodity, each targeting a distinct news angle:
#   1. Futures & price discovery
#   2. Supply chain, exports, trade flows
#   3. Crop conditions, harvest, weather
#   4. Macro shocks — wars, sanctions, pandemics
#   5. Policy — tariffs, USDA reports, government action
#   6. General market news (catch-all)
COMMODITIES = [
    {
        'name': 'wheat',
        'fred_series': 'PWHEAMTUSDM',
        'fred_frequency': 'monthly',
        'serp_queries': [
            'wheat futures prices trading market',
            'wheat exports supply global trade flows',
            'wheat crop harvest yield drought weather',
            'wheat war Ukraine Russia sanctions shortage',
            'wheat USDA report tariff subsidy policy',
            'wheat market news outlook analysis',
        ],
    },
    {
        'name': 'corn',
        'fred_series': 'PMAIZMTUSDM',
        'fred_frequency': 'monthly',
        'serp_queries': [
            'corn futures prices trading market',
            'corn exports supply global trade ethanol',
            'corn crop harvest yield drought weather',
            'corn USDA acreage report production forecast',
            'corn tariff trade war China imports policy',
            'corn market news outlook analysis',
        ],
    },
    {
        'name': 'oil',
        'fred_series': 'DCOILWTICO',
        'fred_frequency': 'daily',
        'serp_queries': [
            'crude oil WTI Brent futures prices trading',
            'oil supply production OPEC output barrels',
            'oil demand inventory stockpile drawdown',
            'oil war sanctions Russia Iran Venezuela geopolitical',
            'oil market outlook forecast energy prices',
            'crude oil news analysis refinery pipeline',
        ],
    },
]

# ── SerpAPI settings ──────────────────────────────────────────────────────────
# More sources = broader coverage across outlets
SERP_SOURCES = [
    'reuters.com',
    'bloomberg.com',
    'cnbc.com',
    'investing.com',
    'ft.com',
    'wsj.com',
    'marketwatch.com',
]
MAX_PAGES_PER_COMBO = 3    # 3 pages × 10 results = up to 30 per (query, source, year)
REQUEST_BUDGET      = 2000  # 3 × 6 × 7 × 17 × 3 = 6426 max combos; budget limits spend
SERP_DELAY_MIN      = 1.0
SERP_DELAY_MAX      = 2.5

# ── Deduplication ─────────────────────────────────────────────────────────────
# Drop on (source, normalized_title) so same story from same outlet is never
# stored twice even if the URL differs across SerpAPI result pages.
def normalize_title(t: str) -> str:
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', '', t.lower())).strip()

# ── FinBERT settings ──────────────────────────────────────────────────────────
FINBERT_MODEL  = 'ProsusAI/finbert'
EMBED_BATCH    = 8
PCA_DIM        = 16
MAX_TOKEN_LEN  = 512

# ── Data directory ────────────────────────────────────────────────────────────
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

DEVICE = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if getattr(torch.backends, 'mps', None)
          and torch.backends.mps.is_available() else torch.device('cpu'))

print(f'✓ Device: {DEVICE}')
print(f'✓ Commodities: {[c["name"] for c in COMMODITIES]}')
print(f'✓ Queries per commodity: {len(COMMODITIES[0]["serp_queries"])}')
print(f'✓ Sources ({len(SERP_SOURCES)}): {SERP_SOURCES}')
print(f'✓ Pages per combo: {MAX_PAGES_PER_COMBO}')
print(f'✓ SerpAPI budget: {REQUEST_BUDGET} requests')

---
## 2 · Folder Structure

In [15]:
# Create commodity subfolders
for cfg in COMMODITIES:
    name = cfg['name']
    commodity_dir = DATA_DIR / name
    commodity_dir.mkdir(exist_ok=True)

print('✓ Folder structure created:')
for cfg in COMMODITIES:
    name = cfg['name']
    commodity_dir = DATA_DIR / name
    print(f'  data/{name}/')
    for f in ['prices', 'news', 'embeddings', 'sentiment']:
        if f == 'prices':
            print(f'    {name}_prices.csv')
        elif f == 'news':
            print(f'    {name}_news.csv')
        elif f == 'embeddings':
            print(f'    {name}_news_embeddings.pt')
        elif f == 'sentiment':
            print(f'    {name}_news_sentiment.csv')

✓ Folder structure created:
  data/wheat/
    wheat_prices.csv
    wheat_news.csv
    wheat_news_embeddings.pt
    wheat_news_sentiment.csv


---
## 3 · FRED Price Data

- **wheat / corn**: `PWHEAMTUSDM` / `PMAIZMTUSDM` — monthly global prices, resampled to business-daily.
- **oil**: `DCOILWTICO` — daily WTI spot price.

Files saved to `data/{commodity}/{commodity}_prices.csv`

In [16]:
fred = Fred(api_key=FRED_API_KEY)

def fetch_fred_prices(series_id: str, frequency: str,
                      start: str, end: str) -> pd.DataFrame:
    raw = fred.get_series(series_id, observation_start=start, observation_end=end)
    df = pd.DataFrame({'Price': raw}).reset_index()
    df.columns = ['Date', 'Price']
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.dropna(subset=['Price']).sort_values('Date')

    if frequency == 'monthly':
        df = (df.set_index('Date')
                .resample('B')
                .ffill()
                .reset_index())

    df = df[(df['Date'] >= START_DATE) & (df['Date'] <= END_DATE)]
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    return df.reset_index(drop=True)

print('Fetching FRED prices...\n')
for cfg in COMMODITIES:
    name = cfg['name']
    out_csv = DATA_DIR / name / f'{name}_prices.csv'

    if out_csv.exists():
        existing = pd.read_csv(out_csv)
        print(f'  {name}: already exists ({len(existing):,} rows) — skipping')
        continue

    df = fetch_fred_prices(cfg['fred_series'], cfg['fred_frequency'],
                           START_DATE, END_DATE)
    df.to_csv(out_csv, index=False)
    print(f'  {name}: {len(df):,} rows saved')

print()

Fetching FRED prices...

  wheat: already exists (4,216 rows) — skipping



---
## 4 · SerpAPI News Collection

**Strategy:** 3 broad queries per commodity × 4 sources × 17 years × 2 pages = up to ~1,200 combos.

- **Broad queries** (no shock-term AND filter) — captures routine market coverage, not just crisis events
- **Pagination** — fetches page 0 + page 1 per (query, source, year) combo
- **Checkpoint** — tracks `{commodity}:q{idx}:{source}:{year}` keys; re-runs skip done combos
- **Budget cap** — `REQUEST_BUDGET = 800` prevents API exhaustion; re-run to continue
- **Deduplication** — URL-based, across all queries for the same commodity

> **Note:** If you ran a previous version, reset the checkpoint with `import json; (Path('data')/'serp_checkpoint.json').write_text(json.dumps({'fetched': {}, 'total_requests': 0}))` to start fresh with the new query keys.

In [ ]:
# ── Reset checkpoint (run once if upgrading from the old single-query version) ─
# The old checkpoint used keys like "wheat:reuters.com:2020".
# The new keys include the query index: "wheat:q0:reuters.com:2020".
# Running this cell wipes the old checkpoint so all combos are fetched fresh.

import json
_ckpt_path = Path('data') / 'serp_checkpoint.json'
_ckpt_path.write_text(json.dumps({'fetched': {}, 'total_requests': 0}, indent=2))
print('✓ Checkpoint reset — all combos will be re-fetched with new query keys')

In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────
CHECKPOINT_FILE = DATA_DIR / 'serp_checkpoint.json'

def load_checkpoint() -> dict:
    if CHECKPOINT_FILE.exists():
        return json.loads(CHECKPOINT_FILE.read_text())
    return {'fetched': {}, 'total_requests': 0}

def save_checkpoint(ckpt: dict):
    CHECKPOINT_FILE.write_text(json.dumps(ckpt, indent=2))

def ckpt_key(commodity: str, query_idx: int, source: str, year: int) -> str:
    return f'{commodity}:q{query_idx}:{source}:{year}'

def existing_urls(csv_path: Path) -> set:
    if csv_path.exists():
        try:
            return set(pd.read_csv(csv_path)['url'].dropna().tolist())
        except Exception:
            pass
    return set()

# ── Date parser ───────────────────────────────────────────────────────────────
_MONTHS = {m: i+1 for i, m in enumerate(
    ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])}

def parse_serp_date(raw: str) -> str:
    if not raw:
        return ''
    if 'ago' in raw.lower():
        return datetime.utcnow().strftime('%Y-%m-%d')
    m = re.search(r'([A-Za-z]+)\s+(\d{1,2}),?\s+(\d{4})', raw)
    if m:
        mon_str, day, year = m.group(1)[:3].capitalize(), int(m.group(2)), int(m.group(3))
        mon = _MONTHS.get(mon_str, 0)
        if mon:
            return f'{year}-{mon:02d}-{day:02d}'
    m2 = re.search(r'(\d{4})-(\d{2})-(\d{2})', raw)
    if m2:
        return m2.group(0)
    return ''

# ── Core SerpAPI fetch (with pagination) ─────────────────────────────────────
def fetch_serp_combo(commodity: str, query: str, query_idx: int,
                     source: str, year: int, ckpt: dict) -> tuple[list[dict], int]:
    """
    Fetch one (commodity, query_idx, source, year) combo with pagination.
    Returns (records, requests_used). requests_used == -1 means budget exhausted.
    """
    key = ckpt_key(commodity, query_idx, source, year)
    if key in ckpt['fetched']:
        return [], 0  # already done

    if ckpt['total_requests'] >= REQUEST_BUDGET:
        return [], -1

    records = []
    requests_used = 0

    for page in range(MAX_PAGES_PER_COMBO):
        if ckpt['total_requests'] >= REQUEST_BUDGET:
            break

        params = {
            'q':       f'site:{source} {query}',
            'api_key': SERP_API_KEY,
            'num':     10,
            'start':   page * 10,
            'tbs':     f'cdr:1,cd_min:01/01/{year},cd_max:12/31/{year}',
        }

        try:
            time.sleep(random.uniform(SERP_DELAY_MIN, SERP_DELAY_MAX))
            result = GoogleSearch(params).get_dict()
            requests_used += 1
            ckpt['total_requests'] += 1

            organic = result.get('organic_results', [])
            if not organic:
                break  # no more results for this year

            for r in organic:
                records.append({
                    'commodity':   commodity,
                    'title':       r.get('title', ''),
                    'date':        parse_serp_date(r.get('date', '')),
                    'source':      source,
                    'description': r.get('snippet', ''),
                    'url':         r.get('link', ''),
                })

            if len(organic) < 10:
                break  # fewer results than requested → last page

        except Exception as e:
            print(f'    [warn] {source} {year} page {page}: {e}')
            break

    ckpt['fetched'][key] = len(records)
    save_checkpoint(ckpt)
    return records, requests_used

print('✓ SerpAPI helpers ready')
ckpt = load_checkpoint()
print(f'  Checkpoint: {ckpt["total_requests"]} requests used, '
      f'{len(ckpt["fetched"])} combos fetched')

In [ ]:
# ── Main news collection loop ────────────────────────────────────────────────
# Iterates: commodity → query (3 angles) → source (4) → year (17) → page (2)
# Checkpoint skips already-fetched combos on re-runs.
ckpt = load_checkpoint()
budget_exhausted = False

print(f'SerpAPI budget: {ckpt["total_requests"]} / {REQUEST_BUDGET}')
print(f'Already fetched combos: {len(ckpt["fetched"])}\n')

for cfg in COMMODITIES:
    if budget_exhausted:
        break

    name    = cfg['name']
    queries = cfg['serp_queries']
    out_csv = DATA_DIR / name / f'{name}_news.csv'
    seen_urls = existing_urls(out_csv)

    print(f'── {name.upper()} (existing: {len(seen_urls):,} URLs) ──')
    commodity_new = 0

    for q_idx, query in enumerate(queries):
        if budget_exhausted:
            break
        print(f'  Query {q_idx+1}/{len(queries)}: {query[:60]}…')

        for source in SERP_SOURCES:
            if budget_exhausted:
                break
            source_new = 0

            for year in YEARS:
                if budget_exhausted:
                    break

                records, req_count = fetch_serp_combo(
                    name, query, q_idx, source, year, ckpt)

                if req_count == -1:
                    budget_exhausted = True
                    break

                if not records:
                    continue

                new_records = [r for r in records
                               if r['url'] not in seen_urls and r['date']]
                seen_urls.update(r['url'] for r in new_records)

                if new_records:
                    df_new = pd.DataFrame(new_records)
                    write_header = not out_csv.exists()
                    df_new.to_csv(out_csv, mode='a', header=write_header, index=False)
                    source_new += len(new_records)
                    commodity_new += len(new_records)

            if source_new:
                print(f'    {source}: +{source_new} new articles')

    total = len(pd.read_csv(out_csv)) if out_csv.exists() else 0
    print(f'  Total: {total:,} articles (+{commodity_new} this run)\n')

print(f'Requests used: {ckpt["total_requests"]} / {REQUEST_BUDGET}')
if budget_exhausted:
    print(f'⚠ Budget limit reached — re-run to continue (already-fetched combos skipped)')

---
## 5 · FinBERT Embeddings & Sentiment

For each commodity:
1. Load news CSV, clean text (title + description)
2. Extract 768-D `[CLS]` embeddings via FinBERT
3. PCA-reduce to 16-D
4. Score sentiment → positive / negative / neutral
5. Mean-pool per calendar day
6. Save:
   - `{commodity}_news_embeddings.pt` — `dict[date_str, Tensor(16,)]`
   - `{commodity}_news_sentiment.csv` — daily sentiment + article count

Skips if output files exist (delete to reprocess).

In [ ]:
# ── Text cleaning ────────────────────────────────────────────────────────────
_BOILERPLATE_RE = re.compile(
    r'^(?:\*\s*)?(?:By\s+[A-Z][a-zA-Z\s\-\']+'
    r'[A-Z]{2,}[\w\s,]*\([^)]+\)\s*[-–—]\s*)?(?:\*\s*)?',
    re.MULTILINE,
)

def clean_text(row: pd.Series) -> str:
    title = str(row.get('title', '')).strip()
    desc = str(row.get('description', '')).strip()
    desc = _BOILERPLATE_RE.sub('', desc).strip()
    if desc.endswith('...'):
        desc = desc[:-3].strip()
    return f'{title}. {desc}' if desc else title

# ── Embedding extraction ──────────────────────────────────────────────────────
def extract_cls_embeddings(texts: list, tokenizer, model, device,
                           batch_size: int = EMBED_BATCH) -> torch.Tensor:
    all_emb = []
    for start in tqdm(range(0, len(texts), batch_size), desc='  Embeddings', leave=False):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_TOKEN_LEN, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        all_emb.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(all_emb, dim=0)

# ── PCA reduction ─────────────────────────────────────────────────────────────
def apply_pca(emb: torch.Tensor, n: int = PCA_DIM) -> torch.Tensor:
    pca = PCA(n_components=n)
    reduced = pca.fit_transform(emb.numpy())
    var = pca.explained_variance_ratio_.sum()
    print(f'  PCA {emb.shape[1]}-D → {n}-D (var: {var:.3f})')
    return torch.from_numpy(reduced).float()

# ── Sentiment extraction ──────────────────────────────────────────────────────
def extract_sentiment(texts: list, tokenizer, sent_model, device,
                      batch_size: int = EMBED_BATCH) -> pd.DataFrame:
    rows = []
    for start in tqdm(range(0, len(texts), batch_size), desc='  Sentiment', leave=False):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_TOKEN_LEN, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            probs = torch.softmax(sent_model(**enc).logits, dim=-1).cpu()
        for row in probs:
            pos, neg, neu = row[0].item(), row[1].item(), row[2].item()
            rows.append({'sentiment_pos': pos, 'sentiment_neg': neg,
                         'sentiment_neu': neu, 'sentiment_score': pos - neg})
    return pd.DataFrame(rows)

# ── Daily aggregation ─────────────────────────────────────────────────────────
def aggregate_daily(df: pd.DataFrame, emb: torch.Tensor) -> dict:
    dates = df['date_day'].tolist()
    idx_map = {}
    for i, d in enumerate(dates):
        idx_map.setdefault(d, []).append(i)
    return {day: torch.mean(emb[idxs], dim=0)
            for day, idxs in sorted(idx_map.items())}

print('✓ FinBERT helpers ready')

In [ ]:
print('Processing FinBERT embeddings & sentiment...\n')

for cfg in COMMODITIES:
    name = cfg['name']
    news_csv = DATA_DIR / name / f'{name}_news.csv'
    emb_path = DATA_DIR / name / f'{name}_news_embeddings.pt'
    sent_path = DATA_DIR / name / f'{name}_news_sentiment.csv'

    if not news_csv.exists():
        print(f'{name}: news CSV not found — run §4 first')
        continue

    if emb_path.exists() and sent_path.exists():
        print(f'{name}: already processed (delete files to reprocess)')
        continue

    print(f'{name.upper()}:')

    # Load & clean
    df = pd.read_csv(news_csv)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    df['date_day'] = df['date'].dt.strftime('%Y-%m-%d')
    df = df[(df['date_day'] >= START_DATE) & (df['date_day'] <= END_DATE)]
    df['clean_text'] = df.apply(clean_text, axis=1)
    df = df.reset_index(drop=True)
    texts = df['clean_text'].tolist()
    print(f'  {len(df):,} articles, {df["date_day"].nunique():,} days')

    # Load FinBERT
    tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
    base_model = AutoModel.from_pretrained(FINBERT_MODEL).to(DEVICE).eval()

    # Extract embeddings
    emb_768 = extract_cls_embeddings(texts, tokenizer, base_model, DEVICE, EMBED_BATCH)
    del base_model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    # PCA
    emb_16 = apply_pca(emb_768, PCA_DIM)

    # Sentiment
    sent_model = AutoModelForSequenceClassification.from_pretrained(
        FINBERT_MODEL).to(DEVICE).eval()
    sent_df = extract_sentiment(texts, tokenizer, sent_model, DEVICE, EMBED_BATCH)
    del sent_model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    df = pd.concat([df.reset_index(drop=True), sent_df], axis=1)

    # Daily aggregation
    daily_emb = aggregate_daily(df, emb_16)
    daily_sent = (df.groupby('date_day')[
                      ['sentiment_pos', 'sentiment_neg', 'sentiment_neu', 'sentiment_score']]
                    .mean()
                    .reset_index()
                    .rename(columns={'date_day': 'date'}))
    daily_sent['article_count'] = df.groupby('date_day').size().values
    daily_sent = daily_sent.sort_values('date').reset_index(drop=True)

    # Save
    torch.save(daily_emb, emb_path)
    daily_sent.to_csv(sent_path, index=False)

    print(f'  Saved: embeddings ({len(daily_emb)} days), sentiment ({len(daily_sent)} rows)')
    print()

---
## 6 · Summary

In [ ]:
print('=' * 70)
print('COMMODITY DATA COLLECTION — FINAL SUMMARY')
print('=' * 70)
print()

ckpt_final = load_checkpoint()
print(f'SerpAPI requests used: {ckpt_final["total_requests"]} / {REQUEST_BUDGET}')
print()

for cfg in COMMODITIES:
    name = cfg['name']
    price_csv = DATA_DIR / name / f'{name}_prices.csv'
    news_csv = DATA_DIR / name / f'{name}_news.csv'
    emb_path = DATA_DIR / name / f'{name}_news_embeddings.pt'
    sent_path = DATA_DIR / name / f'{name}_news_sentiment.csv'

    n_prices = len(pd.read_csv(price_csv)) if price_csv.exists() else 0
    n_news = len(pd.read_csv(news_csv)) if news_csv.exists() else 0
    n_emb = len(torch.load(emb_path, weights_only=False)) if emb_path.exists() else 0
    n_sent = len(pd.read_csv(sent_path)) if sent_path.exists() else 0

    print(f'{name.upper()}')
    print(f'  Prices: {n_prices:,} rows')
    print(f'  News:   {n_news:,} articles  |  Embeddings: {n_emb:,} days  |  Sentiment: {n_sent:,} days')
    print()

print('Files in data/:')
for cfg in COMMODITIES:
    name = cfg['name']
    dir_path = DATA_DIR / name
    if dir_path.exists():
        for f in sorted(dir_path.glob('*')):
            size_kb = f.stat().st_size / 1024
            print(f'  {str(f.relative_to(DATA_DIR)):40s}  {size_kb:8.1f} KB')